# Session 1 — Extras

Dieses Notebook enthält optionale Inhalte für Teilnehmer, die die Kerninhalte aus den Notebooks 1 und 2 bereits abgeschlossen haben.

Die Themen hier sind **nicht Voraussetzung für Session 2** — sie vertiefen aber das Verständnis und machen den Code kompakter und professioneller.

## Inhalte

1. List Comprehensions
2. `dict.get()` — sicherer Zugriff auf Dictionary-Werte
3. Fehlerbehandlung (try / except)
4. Lambda-Funktionen
5. `*args` und `**kwargs`
6. Erweiterte String-Methoden
7. Method Chaining in pandas
8. `.query()` zum Filtern
9. Erweiterte Aggregationen mit `.agg()`
10. Binning mit `pd.cut()` und `pd.qcut()`

---

In [ ]:
import pandas as pd

---
# 1. List Comprehensions

Eine List Comprehension ist eine kompakte Schreibweise für eine `for`-Schleife, die eine neue Liste erzeugt.

```python
# Klassische Schleife:
ergebnis = []
for x in liste:
    ergebnis.append(ausdruck)

# Als List Comprehension:
ergebnis = [ausdruck for x in liste]
```

In [ ]:
# Klassisch: Quadratzahlen berechnen
zahlen = [1, 2, 3, 4, 5]

quadrate_schleife = []
for z in zahlen:
    quadrate_schleife.append(z ** 2)

# Als List Comprehension — eine Zeile:
quadrate_comp = [z ** 2 for z in zahlen]

print(quadrate_schleife)
print(quadrate_comp)  # Identisches Ergebnis

In [ ]:
# Mit Bedingung (if am Ende)
alle_zahlen = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

# Nur gerade Zahlen:
gerade = [z for z in alle_zahlen if z % 2 == 0]
print("Gerade:", gerade)

# Gerade Zahlen quadriert:
gerade_quadrate = [z ** 2 for z in alle_zahlen if z % 2 == 0]
print("Gerade quadriert:", gerade_quadrate)

In [ ]:
# Praxisbeispiel: Texte bereinigen
rohdaten = ["  Hamburg ", "BERLIN", " München ", "KÖLN", "  Frankfurt  "]

# Klassisch:
bereinigt_schleife = []
for s in rohdaten:
    bereinigt_schleife.append(s.strip().title())

# List Comprehension:
bereinigt_comp = [s.strip().title() for s in rohdaten]

print(bereinigt_comp)

In [ ]:
# Dictionary Comprehension — dasselbe Prinzip für Dictionaries
produkte = ["Laptop", "Maus", "Monitor"]
preise   = [999.0,    29.90,   349.0]

# Erstelle ein Dictionary {Produkt: Preis} in einer Zeile
preis_dict = {p: pr for p, pr in zip(produkte, preise)}
print(preis_dict)

### Übung 1

Gegeben ist eine Liste von Beträgen in Euro. Erstelle mit einer List Comprehension eine neue Liste mit den Beträgen in Cent (multipliziert mit 100), aber **nur für Beträge über 50 Euro**.

In [ ]:
betraege = [12.50, 89.99, 3.00, 150.00, 45.00, 200.50, 7.99]

---
## 2. dict.get() — sicherer Zugriff auf Dictionary-Werte

Beim normalen Zugriff mit `dict["key"]` wirft Python einen `KeyError`, wenn der Schlüssel nicht existiert.
`dict.get()` gibt stattdessen `None` zurück — oder einen selbst gewählten Standardwert.

In [ ]:
produkt_preise = {"Laptop": 999.0, "Maus": 29.90, "Monitor": 349.0}

# Normaler Zugriff — KeyError wenn Schlüssel fehlt:
# print(produkt_preise["Tastatur"])  # -> KeyError!

# .get() gibt None zurück, wenn der Schlüssel nicht existiert
preis = produkt_preise.get("Tastatur")
print("Tastatur:", preis)            # None

# Mit Standardwert: wird zurückgegeben, wenn Schlüssel fehlt
preis = produkt_preise.get("Tastatur", 0.0)
print("Tastatur (Standard):", preis) # 0.0

preis = produkt_preise.get("Laptop", 0.0)
print("Laptop:", preis)              # 999.0

---
## 3. Fehlerbehandlung (try / except)

In echten Daten gibt es immer Fehler: leere Felder, falsche Formate, fehlende Werte. Mit `try / except` kann dein Programm mit solchen Fehlern umgehen, ohne abzustürzen.

```python
try:
    # Code, der einen Fehler auslösen könnte
except FehlertypA:
    # Wird ausgeführt, wenn FehlertypA auftritt
except FehlertypB:
    # Wird ausgeführt, wenn FehlertypB auftritt
```

In [ ]:
# Ohne Fehlerbehandlung — würde abstürzen:
# wert = "abc"
# zahl = int(wert)   # ValueError!

# Mit Fehlerbehandlung:
def sicher_in_zahl_umwandeln(wert):
    """Wandelt einen Wert sicher in eine Zahl um."""
    try:
        return float(wert)
    except (ValueError, TypeError):
        print(f"  Warnung: '{wert}' kann nicht umgewandelt werden -> 0")
        return 0.0

# Test mit verschiedenen Eingaben (wie in schmutzigen CSV-Daten)
test_werte = ["123.45", "67", "", "N/A", None, "89.0"]

for wert in test_werte:
    ergebnis = sicher_in_zahl_umwandeln(wert)
    if ergebnis > 0:
        print(f"  OK: {wert!r} -> {ergebnis}")

### Übung 2

Die folgende Liste enthält Bestellungen als Dictionaries. Manche Felder fehlen, manche Werte sind ungültig.

Schreibe eine Funktion `bestellung_bereinigen(d)`, die:
- `dict.get()` benutzt, um fehlende Felder sicher zu lesen (Standard: `"Unbekannt"`, `0`, `0.0`)
- `try / except` benutzt, um Umwandlungsfehler bei `menge` (int) und `preis` (float) abzufangen

Wende die Funktion auf alle Bestellungen an und gib die bereinigten Daten aus.

In [ ]:
bestellungen = [
    {"artikel": "Laptop",   "menge": "2",   "preis": "999.99"},
    {"artikel": "Maus",     "menge": "abc", "preis": "29.90"},
    {"artikel": "Monitor",                  "preis": "349.00"},
    {"artikel": "Tastatur", "menge": "1",   "preis": "N/A"},
]

def bestellung_bereinigen(d):
    """Liest Felder sicher und fängt Umwandlungsfehler ab."""
    artikel = d.get("artikel", "Unbekannt")

    try:
        menge = int(d.get("menge", 0))
    except (ValueError, TypeError):
        menge = 0

    try:
        preis = float(d.get("preis", 0.0))
    except (ValueError, TypeError):
        preis = 0.0

    return {"artikel": artikel, "menge": menge, "preis": preis}

ergebnis = [bestellung_bereinigen(b) for b in bestellungen]
for e in ergebnis:
    print(e)

---
# 4. Lambda-Funktionen

Eine **Lambda-Funktion** ist eine anonyme (namenlose) Einzeilen-Funktion. Sie ist praktisch, wenn man eine einfache Funktion nur einmal, an einer bestimmten Stelle braucht.

```python
# Normale Funktion:
def verdoppeln(x):
    return x * 2

# Lambda-Äquivalent:
verdoppeln = lambda x: x * 2
```

In [ ]:
# Einfaches Beispiel
mwst = lambda netto: round(netto * 1.19, 2)

print(mwst(100))   # 119.0
print(mwst(49.90)) # 59.38

In [ ]:
# Lambda mit mehreren Parametern
rabatt_preis = lambda preis, rabatt_pct: round(preis * (1 - rabatt_pct / 100), 2)

print(rabatt_preis(200, 10))  # 180.0
print(rabatt_preis(50, 20))   # 40.0

In [ ]:
# Häufigster Einsatz: als Argument in sorted(), map(), pandas apply()

kunden = [
    {"name": "Anna",  "umsatz": 4500},
    {"name": "Klaus", "umsatz": 1200},
    {"name": "Sara",  "umsatz": 8900},
]

# Sortieren nach umsatz — ohne Lambda bräuchte man eine separate Hilfsfunktion
sortiert = sorted(kunden, key=lambda k: k["umsatz"], reverse=True)
for k in sortiert:
    print(f"  {k['name']}: {k['umsatz']} Euro")

In [ ]:
# Lambda in pandas: apply() mit inline-Funktion
df = pd.DataFrame({"preis": [100, 250, 80, 430], "kategorie": ["A", "B", "A", "B"]})

# Rabatt: Kategorie A bekommt 10%, B bekommt 15%
df["rabatt_pct"] = df["kategorie"].map({"A": 10, "B": 15})
df["preis_nach_rabatt"] = df.apply(
    lambda row: round(row["preis"] * (1 - row["rabatt_pct"] / 100), 2),
    axis=1
)
df

---
# 5. `*args` und `**kwargs`

Mit `*args` und `**kwargs` kann man Funktionen schreiben, die eine **beliebige Anzahl** von Argumenten akzeptieren.

- `*args` — beliebig viele **positionale** Argumente (als Tuple)
- `**kwargs` — beliebig viele **benannte** Argumente (als Dictionary)

In [ ]:
# *args: beliebig viele Zahlen summieren
def summe(*zahlen):
    print(f"Eingabe: {zahlen}")  # zahlen ist ein Tuple
    return sum(zahlen)

print(summe(1, 2, 3))
print(summe(10, 20, 30, 40, 50))

In [ ]:
# **kwargs: beliebig viele benannte Argumente
def bericht_erstellen(**felder):
    print("=== Bericht ===")
    for schluessel, wert in felder.items():
        print(f"  {schluessel}: {wert}")

bericht_erstellen(name="Anna", abteilung="IT", umsatz=45000, quartal="Q1")

In [ ]:
# Praxisbeispiel: Flexibler CSV-Reader
# Wir wollen pd.read_csv() mit eigenen Standardwerten wrappen,
# aber trotzdem alle Original-Parameter durchreichen können

def csv_laden(dateipfad, **kwargs):
    """Lädt eine CSV mit sinnvollen Standardwerten."""
    # Standardwerte setzen
    defaults = {
        "encoding": "utf-8",
        "sep": ",",
        "on_bad_lines": "warn"
    }
    # Übergebene kwargs überschreiben die defaults
    params = {**defaults, **kwargs}
    print(f"Lade {dateipfad} mit: {params}")
    return pd.read_csv(dateipfad, **params)

# Aufruf — nur Abweichungen vom Standard angeben:
# csv_laden("meine_datei.csv")                        # nutzt alle defaults
# csv_laden("deutsch.csv", sep=";", decimal=",")      # überschreibt sep
print("Funktion definiert — bereit zum Einsatz.")

---
# 6. Erweiterte String-Methoden

Strings haben viele eingebaute Methoden, die beim Bereinigen von Textdaten nützlich sind.

In [ ]:
text = "Umsatz Q1 2024: 45.000 Euro (vorläufig)"

# Enthält einen Substring?
print("vorläufig" in text)           # True
print(text.startswith("Umsatz"))     # True
print(text.endswith(")"))            # True

# Position finden
print(text.find("2024"))             # Index des ersten Vorkommens
print(text.count("0"))               # Wie oft kommt "0" vor?

# Aufteilen und zusammenfügen
teile = text.split(" ")
print(teile)
print(" | ".join(teile[:3]))         # Erste 3 Teile mit " | " verbinden

# Ziffern extrahieren (manuell)
nur_ziffern = "".join(c for c in text if c.isdigit() or c == ".")
print("Ziffern:", nur_ziffern)

In [ ]:
# Pandas .str-Accessor — dieselben Methoden auf ganzen Spalten
df_str = pd.DataFrame({
    "beschreibung": [
        "Laptop Pro 15\" (refurbished)",
        "USB-Maus — kabellos",
        "Monitor 27\" UHD (neu)",
        "Tastatur DE-Layout",
    ]
})

# Enthält "refurbished" oder "neu"?
df_str["ist_neu"]         = df_str["beschreibung"].str.contains("neu", case=False)
df_str["ist_refurbished"] = df_str["beschreibung"].str.contains("refurbished", case=False)

# Klammern-Inhalt extrahieren mit str.extract() und Regex
df_str["zustand"] = df_str["beschreibung"].str.extract(r"\((.+?)\)")

df_str

---
# 7. Method Chaining in pandas

Anstatt viele Zwischenvariablen zu erstellen, kann man pandas-Operationen **verketten** — der Output jeder Operation wird direkt als Input der nächsten verwendet.

Das macht Transformations-Pipelines lesbarer und kürzer.

In [ ]:
df_roh = pd.DataFrame([
    {"name": "  ANNA  ", "abteilung": "vertrieb", "gehalt": 52000, "jahre": 5},
    {"name": "Klaus",    "abteilung": "IT",        "gehalt": 61000, "jahre": 8},
    {"name": "  Sara ",  "abteilung": "vertrieb", "gehalt": 48000, "jahre": 3},
    {"name": "Tom",      "abteilung": "HR",        "gehalt": 44000, "jahre": 6},
    {"name": "Lisa",     "abteilung": "IT",        "gehalt": 67000, "jahre": 11},
])

# OHNE Method Chaining — viele Zwischenvariablen
df1 = df_roh.copy()
df1["name"] = df1["name"].str.strip().str.title()
df1["abteilung"] = df1["abteilung"].str.title()
df1["monatgehalt"] = df1["gehalt"] / 12
df2 = df1[df1["gehalt"] > 45000]
df3 = df2.sort_values("gehalt", ascending=False)
print("Ohne Chaining:")
display(df3)

In [ ]:
# MIT Method Chaining — eine Ausdruckskette, keine Zwischenvariablen
df_clean = (
    df_roh
    .copy()
    .assign(
        name       = lambda df: df["name"].str.strip().str.title(),
        abteilung  = lambda df: df["abteilung"].str.title(),
        monatgehalt= lambda df: df["gehalt"] / 12
    )
    .query("gehalt > 45000")
    .sort_values("gehalt", ascending=False)
    .reset_index(drop=True)
)

print("Mit Method Chaining:")
df_clean

Die Klammern `( ... )` um den gesamten Ausdruck erlauben Zeilenumbrüche ohne Backslash `\`. Das ist der empfohlene Stil.

---
# 8. `.query()` zum Filtern

`.query()` erlaubt es, Filterbedingungen als **lesbaren String** zu schreiben — besonders praktisch bei mehreren Bedingungen.

In [ ]:
df = pd.DataFrame([
    {"name": "Anna",  "abteilung": "Vertrieb", "gehalt": 52000, "jahre": 5},
    {"name": "Klaus", "abteilung": "IT",        "gehalt": 61000, "jahre": 8},
    {"name": "Sara",  "abteilung": "Vertrieb", "gehalt": 48000, "jahre": 3},
    {"name": "Tom",   "abteilung": "HR",        "gehalt": 44000, "jahre": 6},
    {"name": "Lisa",  "abteilung": "IT",        "gehalt": 67000, "jahre": 11},
])

# Klassisch:
klassisch = df[(df["abteilung"] == "IT") & (df["gehalt"] > 60000)]

# Mit .query() — lesbarer:
mit_query = df.query("abteilung == 'IT' and gehalt > 60000")

print("Ergebnis identisch:", klassisch.equals(mit_query))
display(mit_query)

In [ ]:
# Variablen in .query() mit @ einbinden
mindest_gehalt = 50000
abteilungen = ["IT", "Vertrieb"]

ergebnis = df.query("gehalt >= @mindest_gehalt and abteilung in @abteilungen")
ergebnis

---
# 9. Erweiterte Aggregationen mit `.agg()`

`.agg()` erlaubt es, mehrere Aggregationen gleichzeitig und pro Spalte unterschiedlich zu definieren.

In [ ]:
verkauf = pd.DataFrame([
    {"monat": "Jan", "region": "Nord", "umsatz": 12000, "bestellungen": 8},
    {"monat": "Jan", "region": "Süd",  "umsatz": 9500,  "bestellungen": 6},
    {"monat": "Feb", "region": "Nord", "umsatz": 15000, "bestellungen": 10},
    {"monat": "Feb", "region": "Süd",  "umsatz": 11000, "bestellungen": 7},
    {"monat": "Mär", "region": "Nord", "umsatz": 13500, "bestellungen": 9},
    {"monat": "Mär", "region": "Süd",  "umsatz": 10200, "bestellungen": 7},
])

# Verschiedene Aggregationen je Spalte
ergebnis = verkauf.groupby("region").agg(
    umsatz_gesamt   = ("umsatz",      "sum"),
    umsatz_avg      = ("umsatz",      "mean"),
    umsatz_max      = ("umsatz",      "max"),
    bestellungen_sum= ("bestellungen","sum"),
    monate_anzahl   = ("monat",       "count"),
).round(0)

ergebnis

In [ ]:
# Eigene Aggregationsfunktion
def bereich(serie):
    """Berechnet Spannweite (Max - Min)."""
    return serie.max() - serie.min()

verkauf.groupby("region").agg(
    umsatz_spannweite=("umsatz", bereich),
    umsatz_median    =("umsatz", "median"),
)

---
# 10. Binning mit `pd.cut()` und `pd.qcut()`

Kontinuierliche Werte in Kategorien ("Bins") einteilen — z.B. Altersgruppen, Umsatzklassen.

In [ ]:
kunden = pd.DataFrame({
    "name":    ["A","B","C","D","E","F","G","H"],
    "umsatz":  [150, 890, 4200, 320, 7800, 1100, 550, 9500],
    "alter":   [22, 35, 58, 41, 29, 67, 44, 31],
})

# pd.cut(): Feste Grenzen selbst festlegen
kunden["umsatz_klasse"] = pd.cut(
    kunden["umsatz"],
    bins=[0, 500, 2000, 5000, float("inf")],
    labels=["Niedrig", "Mittel", "Hoch", "Premium"]
)

# pd.qcut(): Quantile — gleich viele Einträge in jedem Bin
kunden["alter_quartil"] = pd.qcut(
    kunden["alter"],
    q=4,
    labels=["Q1", "Q2", "Q3", "Q4"]
)

kunden

In [ ]:
# Verteilung der Klassen
print("Umsatzklassen:")
print(kunden["umsatz_klasse"].value_counts().sort_index())

print("\nAlters-Quartile:")
print(kunden["alter_quartil"].value_counts().sort_index())

---
### Abschluss-Übung

Kombiniere mehrere Techniken aus diesem Notebook:

Gegeben ist eine Rohversion einer Mitarbeiterliste. Erstelle mit **Method Chaining** eine bereinigte Auswertungstabelle:

1. Namen bereinigen (strip + title)
2. Abteilung vereinheitlichen (lowercase)
3. Neue Spalte `gehalt_klasse` mit `pd.cut()`: unter 45k = "Junior", 45k–60k = "Senior", über 60k = "Lead"
4. Nur Mitarbeiter mit mehr als 3 Jahren Erfahrung behalten (`.query()`)
5. Nach Gehalt absteigend sortieren
6. Aggregation: Durchschnittsgehalt und Anzahl pro `gehalt_klasse`

In [ ]:
df_ma = pd.DataFrame([
    {"name": " ANNA SCHMIDT ", "abteilung": "Vertrieb", "gehalt": 52000, "jahre": 5},
    {"name": "Klaus Müller",   "abteilung": "IT",       "gehalt": 61000, "jahre": 8},
    {"name": " sara wagner ",  "abteilung": "VERTRIEB", "gehalt": 48000, "jahre": 3},
    {"name": "Tom Becker",     "abteilung": "hr",       "gehalt": 44000, "jahre": 6},
    {"name": "LISA BRAUN",     "abteilung": "IT",       "gehalt": 67000, "jahre": 11},
    {"name": "Marc Schäfer",   "abteilung": "HR",       "gehalt": 46000, "jahre": 2},
])

In [ ]:
# LÖSUNG
df_bereinigt = (
    df_ma
    .copy()
    .assign(
        name       = lambda df: df["name"].str.strip().str.title(),
        abteilung  = lambda df: df["abteilung"].str.lower(),
        gehalt_klasse = lambda df: pd.cut(
            df["gehalt"],
            bins=[0, 45000, 60000, float("inf")],
            labels=["Junior", "Senior", "Lead"]
        )
    )
    .query("jahre > 3")
    .sort_values("gehalt", ascending=False)
    .reset_index(drop=True)
)

print("Bereinigte Mitarbeiterliste:")
display(df_bereinigt)

print("\nDurchschnittsgehalt und Anzahl pro Klasse:")
display(
    df_bereinigt.groupby("gehalt_klasse", observed=True).agg(
        anzahl      =("name",   "count"),
        gehalt_avg  =("gehalt", "mean"),
    ).round(0)
)

---
# Musterlösungen — Weitere Übungsaufgaben

### Aufgabe 1 — List Comprehension & Lambda

Gegeben ist eine Liste von Wörtern.
1. Erstelle mit einer List Comprehension eine neue Liste: nur Wörter mit mehr als 4 Buchstaben, in Großbuchstaben, alphabetisch sortiert.
2. Sortiere die Originalliste nach Wortlänge (kürzeste zuerst) mit `sorted()` und einer Lambda-Funktion.

In [ ]:
# LÖSUNG
woerter = ['Hund', 'Katze', 'Elefant', 'Maus', 'Giraffe', 'Hai', 'Pinguin']

lang_gross  = sorted([w.upper() for w in woerter if len(w) > 4])
nach_laenge = sorted(woerter, key=lambda w: len(w))

print('Lange Wörter (Großbuchstaben):', lang_gross)
print('Nach Länge sortiert:',           nach_laenge)


### Aufgabe 2 — Method Chaining Pipeline

Baue mit Method Chaining eine Transformations-Pipeline:
1. `stadt` und `kategorie` bereinigen (strip + title)
2. `umsatz` auf 2 Nachkommastellen runden
3. Neue Spalte `umsatz_klasse`: `'A'` (≥ 10.000), `'B'` (5.000–9.999), `'C'` (< 5.000)
4. Nur Klasse A und B behalten
5. Nach `umsatz` absteigend sortieren

In [ ]:
# LÖSUNG
import pandas as pd

df_roh = pd.DataFrame([
    {'stadt': '  BERLIN  ', 'kategorie': 'elektronik', 'umsatz': 12450.678},
    {'stadt': 'Hamburg',    'kategorie': 'ZUBEHÖR',    'umsatz': 4890.5},
    {'stadt': ' münchen ', 'kategorie': 'Software',   'umsatz': 9100.123},
    {'stadt': 'KÖLN',       'kategorie': 'elektronik', 'umsatz': 15200.0},
    {'stadt': 'Frankfurt',  'kategorie': 'zubehör',    'umsatz': 3200.9},
])

ergebnis = (
    df_roh.copy()
    .assign(
        stadt         = lambda df: df['stadt'].str.strip().str.title(),
        kategorie     = lambda df: df['kategorie'].str.strip().str.title(),
        umsatz        = lambda df: df['umsatz'].round(2),
        umsatz_klasse = lambda df: pd.cut(
            df['umsatz'], bins=[0, 5000, 10000, float('inf')],
            labels=['C', 'B', 'A'])
    )
    .query("umsatz_klasse in ['A', 'B']")
    .sort_values('umsatz', ascending=False)
    .reset_index(drop=True)
)
display(ergebnis)
